# Notebook 3 — MapReduce con Python multiprocessing

**Proyecto:** Sistema Web de Gestión de Contratación Docente — EMI Cochabamba

Implementa el paradigma MapReduce real sobre el historial de contratos usando `multiprocessing`. No es simulado — usa procesos reales del SO en paralelo.

### Tareas implementadas
1. Monto total y promedio por asignatura
2. Contratos y monto por docente
3. Distribucion por area tematica
4. Benchmark secuencial vs paralelo

In [ ]:
!pip install pandas matplotlib seaborn unidecode -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import multiprocessing as mp
import time
import warnings
from collections import defaultdict
from unidecode import unidecode

warnings.filterwarnings('ignore')

COLORES = {
    'primario':    '#1a4fa0',
    'secundario':  '#2563eb',
    'acento':      '#16a34a',
    'peligro':     '#c0392b',
    'advertencia': '#d97706',
    'morado':      '#7c3aed',
}
PALETA = list(COLORES.values())

plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

N_WORKERS = mp.cpu_count()
print(f'CPUs disponibles: {N_WORKERS}')

## 1. Carga del dataset

In [ ]:
from google.colab import files
print('Sube emi_contratos_limpio.csv (exportado del Notebook 1)')
files.upload()

In [ ]:
df = pd.read_csv('emi_contratos_limpio.csv')

def inferir_area(asignatura):
    a = unidecode(str(asignatura).lower())
    if any(k in a for k in ['red', 'administracion de red']): return 'REDES'
    if any(k in a for k in ['base de dato']): return 'BASES_DE_DATOS'
    if any(k in a for k in ['program', 'python', 'java', 'software', 'web']): return 'PROGRAMACION'
    if any(k in a for k in ['inteligencia']): return 'INTELIGENCIA_ARTIFICIAL'
    if any(k in a for k in ['seguridad', 'forense']): return 'SEGURIDAD'
    if any(k in a for k in ['sistema operativo', 'arquitectura', 'digital']): return 'SISTEMAS'
    if any(k in a for k in ['calculo', 'algebra', 'estadistica', 'ecuacion', 'fisica', 'probabilidad', 'estocastico']): return 'MATEMATICAS'
    if any(k in a for k in ['estructura de dato', 'algoritmo']): return 'ALGORITMOS'
    return 'GESTION'

if 'AREA' not in df.columns:
    df['AREA'] = df['ASIGNATURA'].apply(inferir_area)

registros = df.to_dict('records')
print(f'Dataset: {len(registros)} registros')

## 2. Framework MapReduce con multiprocessing real

Fases: **Map** (paralela) → **Shuffle** (agrupacion) → **Reduce** (paralela)

In [ ]:
def dividir_en_chunks(datos, n_chunks):
    tam = max(1, len(datos) // n_chunks)
    return [datos[i:i+tam] for i in range(0, len(datos), tam)]

def map_worker(args):
    chunk, map_fn = args
    resultado = []
    for registro in chunk:
        pares = map_fn(registro)
        resultado.extend(pares)
    return resultado

def fase_shuffle(pares_map):
    agrupado = defaultdict(list)
    for clave, valor in pares_map:
        agrupado[clave].append(valor)
    return dict(agrupado)

def reduce_worker(args):
    clave, valores, reduce_fn = args
    return (clave, reduce_fn(valores))

def mapreduce(datos, map_fn, reduce_fn, n_workers=None):
    if n_workers is None:
        n_workers = mp.cpu_count()

    t0 = time.perf_counter()

    chunks = dividir_en_chunks(datos, n_workers)
    args_map = [(chunk, map_fn) for chunk in chunks]
    with mp.Pool(processes=n_workers) as pool:
        resultados_map = pool.map(map_worker, args_map)
    pares_map = [par for sub in resultados_map for par in sub]
    t_map = time.perf_counter()

    agrupado = fase_shuffle(pares_map)
    t_shuffle = time.perf_counter()

    args_reduce = [(k, v, reduce_fn) for k, v in agrupado.items()]
    with mp.Pool(processes=n_workers) as pool:
        resultados_reduce = pool.map(reduce_worker, args_reduce)
    t_reduce = time.perf_counter()

    resultado_final = dict(resultados_reduce)

    print(f'  MAP    : {(t_map - t0)*1000:.2f} ms  ({len(pares_map)} pares)')
    print(f'  SHUFFLE: {(t_shuffle - t_map)*1000:.2f} ms  ({len(agrupado)} claves)')
    print(f'  REDUCE : {(t_reduce - t_shuffle)*1000:.2f} ms')
    print(f'  TOTAL  : {(t_reduce - t0)*1000:.2f} ms | Workers: {n_workers}')

    return resultado_final

print('Framework MapReduce listo')

## 3. Tarea 1 — Monto total por asignatura

In [ ]:
def map_monto_asignatura(registro):
    return [(registro['ASIGNATURA'], float(registro['MONTO']))]

def reduce_suma(valores):
    return {
        'total':     sum(valores),
        'promedio':  sum(valores) / len(valores),
        'contratos': len(valores),
        'minimo':    min(valores),
        'maximo':    max(valores),
    }

print('='*60)
print('TAREA 1: Monto total por asignatura')
print('='*60)
resultado_montos = mapreduce(registros, map_monto_asignatura, reduce_suma)

df_montos = pd.DataFrame(resultado_montos).T
df_montos = df_montos.astype({'total': float, 'promedio': float, 'contratos': int})
df_montos = df_montos.sort_values('total', ascending=False)
print()
print(df_montos.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
top = df_montos.head(12)
colors = [COLORES['primario'] if i < 3 else COLORES['secundario'] if i < 6 else '#93c5fd' for i in range(len(top))]
bars = ax.barh(top.index[::-1], top['total'][::-1], color=colors[::-1], alpha=0.9)
for bar, val in zip(bars, top['total'][::-1]):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2, f'Bs. {val:,.0f}', va='center', fontsize=8)
ax.set_title('Monto Total por Asignatura (MapReduce)', fontweight='bold')
ax.set_xlabel('Monto Total (Bs.)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax = axes[1]
top_c = df_montos.sort_values('contratos', ascending=False).head(12)
bars = ax.barh(top_c.index[::-1], top_c['contratos'][::-1], color=COLORES['acento'], alpha=0.85)
for bar, val in zip(bars, top_c['contratos'][::-1]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, str(int(val)), va='center', fontsize=9, fontweight='bold')
ax.set_title('Contratos por Asignatura (MapReduce)', fontweight='bold')
ax.set_xlabel('Numero de Contratos')

plt.tight_layout()
plt.savefig('mapreduce_asignaturas.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Tarea 2 — Contratos por docente

In [ ]:
def map_contratos_docente(registro):
    return [(registro['CEDULA'], {
        'nombre':     registro['NOMBRE'],
        'monto':      float(registro['MONTO']),
        'asignatura': registro['ASIGNATURA'],
    })]

def reduce_docente(valores):
    montos = [v['monto'] for v in valores]
    asignaturas = list({v['asignatura'] for v in valores})
    return {
        'nombre':      valores[0]['nombre'],
        'n_contratos': len(valores),
        'monto_total': sum(montos),
        'asignaturas': len(asignaturas),
    }

print('='*60)
print('TAREA 2: Contratos y monto total por docente')
print('='*60)
resultado_docentes = mapreduce(registros, map_contratos_docente, reduce_docente)

df_doc_mr = pd.DataFrame(resultado_docentes).T
df_doc_mr['n_contratos'] = df_doc_mr['n_contratos'].astype(int)
df_doc_mr['monto_total'] = df_doc_mr['monto_total'].astype(float)
df_doc_mr = df_doc_mr.sort_values('monto_total', ascending=False)
print()
print('Top 15 docentes por monto total:')
print(df_doc_mr[['nombre','n_contratos','monto_total','asignaturas']].head(15).to_string())

## 5. Tarea 3 — Distribucion por area tematica

In [ ]:
def map_area(registro):
    return [(registro['AREA'], {
        'monto':     float(registro['MONTO']),
        'modalidad': registro['MODALIDAD'],
        'cedula':    registro['CEDULA'],
    })]

def reduce_area(valores):
    montos = [v['monto'] for v in valores]
    return {
        'contratos':   len(valores),
        'monto_total': sum(montos),
        'monto_prom':  sum(montos)/len(montos),
        'docentes':    len({v['cedula'] for v in valores}),
        'teoria':      sum(1 for v in valores if v['modalidad'] == 'TEORIA'),
        'laboratorio': sum(1 for v in valores if v['modalidad'] == 'LABORATORIO'),
    }

print('='*60)
print('TAREA 3: Distribucion por area tematica')
print('='*60)
resultado_areas = mapreduce(registros, map_area, reduce_area)

df_areas_mr = pd.DataFrame(resultado_areas).T.astype({'contratos': int, 'docentes': int, 'teoria': int, 'laboratorio': int})
df_areas_mr['monto_total'] = df_areas_mr['monto_total'].astype(float)
df_areas_mr = df_areas_mr.sort_values('contratos', ascending=False)
print()
print(df_areas_mr.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = np.arange(len(df_areas_mr))
w = 0.35
ax.bar(x - w/2, df_areas_mr['teoria'],      w, label='Teoria',      color=COLORES['primario'], alpha=0.85)
ax.bar(x + w/2, df_areas_mr['laboratorio'], w, label='Laboratorio', color=COLORES['acento'],   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_areas_mr.index, rotation=35, ha='right', fontsize=8)
ax.set_title('Contratos Teoria vs Laboratorio por Area (MapReduce)', fontweight='bold')
ax.set_ylabel('Contratos')
ax.legend()

ax = axes[1]
wedges, texts, autotexts = ax.pie(
    df_areas_mr['monto_total'], labels=df_areas_mr.index,
    autopct='%1.1f%%', colors=PALETA[:len(df_areas_mr)], startangle=90, pctdistance=0.8
)
for t in texts: t.set_fontsize(8)
for at in autotexts: at.set_fontsize(7)
ax.set_title('Distribucion Monto Total por Area (MapReduce)', fontweight='bold')

plt.tight_layout()
plt.savefig('mapreduce_areas.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Benchmark: Secuencial vs Paralelo

In [ ]:
def secuencial_monto_asignatura(datos):
    acum = defaultdict(list)
    for r in datos:
        acum[r['ASIGNATURA']].append(float(r['MONTO']))
    return {k: sum(v) for k, v in acum.items()}

tamanos    = [400, 800, 2000, 4000, 8000]
t_seq_list = []
t_par_list = []
n_workers  = mp.cpu_count()

for tam in tamanos:
    repeticiones = (tam // len(registros)) + 1
    datos_test   = (registros * repeticiones)[:tam]

    t0 = time.perf_counter()
    for _ in range(3):
        secuencial_monto_asignatura(datos_test)
    t_seq_list.append((time.perf_counter() - t0) / 3 * 1000)

    t0 = time.perf_counter()
    for _ in range(3):
        chunks   = dividir_en_chunks(datos_test, n_workers)
        args_mp  = [(c, map_monto_asignatura) for c in chunks]
        with mp.Pool(processes=n_workers) as pool:
            res_map = pool.map(map_worker, args_mp)
        pares  = [p for sub in res_map for p in sub]
        agrup  = fase_shuffle(pares)
        args_r = [(k, v, reduce_suma) for k, v in agrup.items()]
        with mp.Pool(processes=n_workers) as pool:
            pool.map(reduce_worker, args_r)
    t_par_list.append((time.perf_counter() - t0) / 3 * 1000)

    print(f'Tam {tam:>5}: Seq={t_seq_list[-1]:.2f}ms  |  MapReduce={t_par_list[-1]:.2f}ms')

speedups = [s/p if p > 0 else 1 for s, p in zip(t_seq_list, t_par_list)]
print(f'Speedups: {[round(s,2) for s in speedups]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(tamanos, t_seq_list, 'o-', color=COLORES['peligro'], linewidth=2.5, markersize=7, label='Secuencial')
ax.plot(tamanos, t_par_list, 's--', color=COLORES['primario'], linewidth=2.5, markersize=7, label=f'MapReduce ({n_workers} workers)')
ax.fill_between(tamanos, t_seq_list, t_par_list, alpha=0.1, color=COLORES['acento'], label='Ganancia paralela')
ax.set_title('Tiempo de Ejecucion: Secuencial vs MapReduce', fontweight='bold')
ax.set_xlabel('Tamano del dataset (registros)')
ax.set_ylabel('Tiempo (ms)')
ax.legend(fontsize=9)
ax.set_xticks(tamanos)

ax = axes[1]
bars = ax.bar(range(len(tamanos)), speedups,
              color=[COLORES['acento'] if s >= 1 else COLORES['peligro'] for s in speedups],
              alpha=0.85, width=0.6)
ax.axhline(1.0, color=COLORES['peligro'], linestyle='--', linewidth=1.5, label='Speedup = 1x')
for bar, s in zip(bars, speedups):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{s:.2f}x',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(tamanos)))
ax.set_xticklabels([str(t) for t in tamanos])
ax.set_title(f'Speedup MapReduce vs Secuencial ({n_workers} workers)', fontweight='bold')
ax.set_xlabel('Tamano del dataset')
ax.set_ylabel('Speedup (veces mas rapido)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('mapreduce_speedup.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Exportar resultados

In [ ]:
df_montos.to_csv('mapreduce_montos_asignatura.csv', encoding='utf-8-sig')
df_doc_mr.to_csv('mapreduce_contratos_docente.csv', encoding='utf-8-sig')
df_areas_mr.to_csv('mapreduce_areas.csv', encoding='utf-8-sig')

from google.colab import files
for archivo in ['mapreduce_montos_asignatura.csv', 'mapreduce_areas.csv',
                'mapreduce_speedup.png', 'mapreduce_asignaturas.png', 'mapreduce_areas.png']:
    files.download(archivo)

print('Resultados exportados')